# Banana ripeness classification with image processing + SVM

This notebook deliberately avoids CNNs and transfer learning. It audits the downloaded images, applies conservative preprocessing, segments the banana using adaptive **CIELAB K-means color segmentation**, extracts interpretable color/texture/shape features, and trains an RBF SVM. It also performs a no-segmentation ablation so the effect of segmentation is measured rather than assumed.

Dataset: [Banalyzer - Banana Ripeness Classification Dataset](https://www.kaggle.com/datasets/iamchaarles/banalyzer-banana-ripeness-classification-dataset)

In [ ]:
# Colab setup
!pip -q install kagglehub opencv-python-headless scikit-image scikit-learn seaborn joblib tqdm

from pathlib import Path
import hashlib, json, math, os, re, warnings
import cv2
import joblib
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image, ImageOps
from scipy.stats import skew
from skimage.feature import graycomatrix, graycoprops, local_binary_pattern
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             classification_report, confusion_matrix, f1_score)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
cv2.setRNGSeed(RANDOM_STATE)
sns.set_theme(style='whitegrid')


## 1. Download and discover the real folder layout

The dataset's test folder uses names such as `unripe_test` and `over_ripe_test`, whereas training folders use the class name alone. The discovery code below normalizes both conventions.

In [ ]:
DATASET_ROOT = Path(kagglehub.dataset_download(
    'iamchaarles/banalyzer-banana-ripeness-classification-dataset'
))
print('Dataset location:', DATASET_ROOT)

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
CLASS_ORDER = ['unripe', 'ripe', 'overripe', 'rotten']

def normalize_label_from_path(path):
    # Search nearest parent first. Check overripe before ripe because 'ripe' is a substring.
    for part in reversed(path.parts[:-1]):
        token = re.sub(r'[^a-z]', '', part.lower())
        if 'overripe' in token: return 'overripe'
        if 'unripe' in token: return 'unripe'
        if 'rotten' in token: return 'rotten'
        if token == 'ripe' or token.startswith('ripetest'): return 'ripe'
    return None

def infer_split(path):
    parts = [re.sub(r'[^a-z]', '', p.lower()) for p in path.parts]
    if any(p == 'test' or p.endswith('test') for p in parts): return 'test'
    if any(p == 'train' or p.endswith('train') for p in parts): return 'train'
    if any(p in {'val', 'valid', 'validation'} for p in parts): return 'val'
    return 'unspecified'

rows = []
for p in DATASET_ROOT.rglob('*'):
    if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
        label = normalize_label_from_path(p)
        if label is not None:
            rows.append({'path': str(p), 'label': label, 'split': infer_split(p)})

records = pd.DataFrame(rows)
assert len(records), 'No labeled images found. Inspect DATASET_ROOT and folder names.'
records['label'] = pd.Categorical(records['label'], categories=CLASS_ORDER, ordered=True)
display(pd.crosstab(records['label'], records['split'], margins=True))
print('Total labeled images:', len(records))
print('Example path:', records.iloc[0].path)


## 2. Audit size, lighting, focus, and noise

Resize is selected from the actual lower-quartile image size, capped at 256 px and rounded to a multiple of 32. Aspect ratio is preserved with padding. This avoids stretching the banana and avoids needlessly upscaling small images. Noise is estimated as the mean absolute residual from a 3×3 median filter; focus uses variance of the Laplacian.

In [ ]:
def read_rgb(path):
    # EXIF transpose fixes phone images whose orientation is stored as metadata.
    with Image.open(path) as im:
        return np.asarray(ImageOps.exif_transpose(im).convert('RGB'))

audit_rows, bad_files = [], []
for row in tqdm(records.itertuples(index=False), total=len(records), desc='Auditing'):
    try:
        rgb = read_rgb(row.path)
        h, w = rgb.shape[:2]
        scale = min(1.0, 256 / max(h, w))
        small = cv2.resize(rgb, (max(1, round(w*scale)), max(1, round(h*scale))),
                           interpolation=cv2.INTER_AREA)
        gray = cv2.cvtColor(small, cv2.COLOR_RGB2GRAY)
        residual = cv2.absdiff(gray, cv2.medianBlur(gray, 3))
        audit_rows.append({
            'path': row.path, 'width': w, 'height': h, 'aspect': w/h,
            'brightness': float(gray.mean()),
            'contrast': float(gray.std()),
            'focus_laplacian': float(cv2.Laplacian(gray, cv2.CV_64F).var()),
            'noise_residual': float(residual.mean())
        })
    except Exception as exc:
        bad_files.append((row.path, str(exc)))

audit = pd.DataFrame(audit_rows)
records = records[~records.path.isin([p for p, _ in bad_files])].reset_index(drop=True)
display(audit[['width','height','aspect','brightness','contrast',
               'focus_laplacian','noise_residual']].describe(percentiles=[.1,.25,.5,.75,.9]).round(2))
print('Unreadable images:', len(bad_files))

short_sides = np.minimum(audit.width, audit.height)
q25 = float(np.percentile(short_sides, 25))
TARGET_SIDE = int(np.clip(math.floor(q25 / 32) * 32, 160, 256))
print(f'Chosen working size: {TARGET_SIDE} x {TARGET_SIDE} (25th percentile short side={q25:.0f}px)')
print('Preprocessing: aspect-preserving resize + mild bilateral denoising + LAB-luminance CLAHE.')


In [ ]:
# Exact train/test duplicate audit. Remove test duplicates from evaluation if any are found.
def file_sha1(path, block=1 << 20):
    digest = hashlib.sha1()
    with open(path, 'rb') as f:
        while chunk := f.read(block): digest.update(chunk)
    return digest.hexdigest()

records['sha1'] = [file_sha1(p) for p in tqdm(records.path, desc='Duplicate check')]
train_hashes = set(records.loc[records.split == 'train', 'sha1'])
leaked = (records.split == 'test') & records.sha1.isin(train_hashes)
print('Exact duplicates crossing train -> test:', int(leaked.sum()))
if leaked.any():
    print('Removing those test copies so evaluation is not inflated.')
    records = records.loc[~leaked].reset_index(drop=True)


## 3. Preprocessing and adaptive color segmentation

Why this method:

- A fixed HSV yellow/green threshold is unsuitable because the target classes deliberately include brown and nearly black peel.
- CIELAB makes color differences more perceptually meaningful and separates luminance from chroma.
- K-means adapts to each image's illumination/background; spatial coordinates discourage scattered color matches. Clusters must overlap banana-color seeds (green, yellow, brown, or dark peel), while border-dominant and bright-neutral clusters are treated as background.
- Morphology removes isolated noise and closes small gaps. Seeded GrabCut optionally refines the color mask and recovers dark spots enclosed by the banana without selecting a pale cutting board.
- CLAHE is applied only to LAB luminance, so chromatic ripeness cues are not independently equalized or recolored.

In [ ]:
def resize_with_padding(rgb, side):
    h, w = rgb.shape[:2]
    scale = side / max(h, w)
    nw, nh = max(1, round(w*scale)), max(1, round(h*scale))
    interp = cv2.INTER_AREA if scale < 1 else cv2.INTER_CUBIC
    resized = cv2.resize(rgb, (nw, nh), interpolation=interp)
    # Use the image-border median instead of a hard-coded black/white pad.
    border = np.concatenate([resized[0], resized[-1], resized[:,0], resized[:,-1]], axis=0)
    pad_color = tuple(np.median(border, axis=0).astype(np.uint8).tolist())
    top, bottom = (side-nh)//2, side-nh-(side-nh)//2
    left, right = (side-nw)//2, side-nw-(side-nw)//2
    return cv2.copyMakeBorder(resized, top, bottom, left, right,
                              cv2.BORDER_CONSTANT, value=pad_color)

def preprocess(rgb, side=TARGET_SIDE):
    rgb = resize_with_padding(rgb, side)
    # Mild edge-preserving denoising: reduces sensor/JPEG noise without erasing peel boundaries.
    rgb = cv2.bilateralFilter(rgb, d=5, sigmaColor=25, sigmaSpace=25)
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    L, a, b = cv2.split(lab)
    L = cv2.createCLAHE(clipLimit=1.5, tileGridSize=(8,8)).apply(L)
    enhanced = cv2.cvtColor(cv2.merge([L, a, b]), cv2.COLOR_LAB2RGB)
    return enhanced

def fill_holes(mask):
    inv = cv2.bitwise_not(mask)
    flood = inv.copy()
    padded = np.zeros((mask.shape[0]+2, mask.shape[1]+2), np.uint8)
    cv2.floodFill(flood, padded, (0,0), 0)
    return cv2.bitwise_or(mask, flood)

def clean_mask(mask, max_components=6, reject_border=True):
    h, w = mask.shape
    k = max(3, int(round(min(h,w)*0.02)) | 1)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k,k))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    n, labels, stats, centroids = cv2.connectedComponentsWithStats(mask, 8)
    if n <= 1: return mask
    cy, cx = h/2, w/2
    candidates, interior_candidates = [], []
    for i in range(1, n):
        area = stats[i, cv2.CC_STAT_AREA]
        dist = np.hypot((centroids[i][0]-cx)/w, (centroids[i][1]-cy)/h)
        if area >= 0.003*h*w:
            score = area * np.exp(-2*dist)
            candidates.append((score, i))
            component = labels == i
            touches = (component[0].any() or component[-1].any() or
                       component[:,0].any() or component[:,-1].any())
            if not touches: interior_candidates.append((score, i))
    pool = interior_candidates if reject_border and interior_candidates else candidates
    keep = [i for _, i in sorted(pool, reverse=True)[:max_components]]
    if not keep: keep = [1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))]
    out = np.isin(labels, keep).astype(np.uint8)*255
    return fill_holes(out)

def banana_color_seeds(rgb):
    # Broad priors cover all four stages. They initialize segmentation; they are not labels.
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    H, S, V = cv2.split(hsv)
    L = lab[...,0]
    green = (H >= 25) & (H <= 95) & (S >= 45) & (V >= 30)
    yellow = (H >= 14) & (H <= 39) & (S >= 65) & (V >= 65)
    brown = (H <= 24) & (S >= 55) & (V >= 20) & (V <= 190)
    dark_peel = (V <= 85) & (L <= 100)
    # Choose the dominant fresh-peel family when it occupies a meaningful area.
    # This stops a brown wooden table from becoming foreground in a green-banana image.
    green_fraction, yellow_fraction = green.mean(), yellow.mean()
    if green_fraction >= 0.035 and green_fraction >= 0.75*yellow_fraction:
        primary = green | yellow
    elif yellow_fraction >= 0.035:
        primary = yellow | green
    else:
        primary = brown | dark_peel
    radius = max(7, int(round(min(rgb.shape[:2])*0.055)) | 1)
    near_primary = cv2.dilate(primary.astype(np.uint8)*255,
                              cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(radius,radius)),
                              iterations=2) > 0
    if green_fraction >= 0.035 or yellow_fraction >= 0.035:
        seed_bool = primary | ((brown | dark_peel) & near_primary)
    else:
        seed_bool = primary
    seed = seed_bool.astype(np.uint8)*255
    k = max(3, int(round(min(rgb.shape[:2])*0.012)) | 1)
    seed = cv2.morphologyEx(seed, cv2.MORPH_OPEN,
                            cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(k,k)))
    return seed

def lab_kmeans_segmentation(rgb, use_grabcut=True):
    h, w = rgb.shape[:2]
    seg_scale = min(1.0, 160/max(h,w))
    sw, sh = max(32, round(w*seg_scale)), max(32, round(h*seg_scale))
    small = cv2.resize(rgb, (sw, sh), interpolation=cv2.INTER_AREA)
    color_seed_small = banana_color_seeds(small) > 0
    lab = cv2.cvtColor(small, cv2.COLOR_RGB2LAB).astype(np.float32)
    yy, xx = np.mgrid[0:sh, 0:sw].astype(np.float32)
    # Normalized LAB plus lightly weighted x/y = color-led, spatially coherent clusters.
    samples = np.column_stack([
        lab[...,0].ravel()/255, (lab[...,1].ravel()-128)/128,
        (lab[...,2].ravel()-128)/128, 0.18*xx.ravel()/sw, 0.18*yy.ravel()/sh
    ]).astype(np.float32)
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.01)
    _, labels, _ = cv2.kmeans(samples, 5, None, criteria, 3, cv2.KMEANS_PP_CENTERS)
    labels = labels.reshape(sh, sw)

    ring = max(2, round(min(sh,sw)*0.08))
    border = np.zeros((sh,sw), bool)
    border[:ring] = border[-ring:] = True
    border[:,:ring] = border[:,-ring:] = True
    center = (((xx-sw/2)/(0.42*sw))**2 + ((yy-sh/2)/(0.42*sh))**2) <= 1
    cluster_scores, cluster_border_shares = [], []
    for k in range(5):
        cluster = labels == k
        seed_overlap = color_seed_small[cluster].mean() if cluster.any() else 0
        border_share = (labels[border] == k).mean()
        center_share = (labels[center] == k).mean()
        score = 2.4*seed_overlap + 0.35*center_share - 1.25*border_share
        cluster_scores.append(score)
        cluster_border_shares.append(border_share)
    # A cluster needs genuine peel-color support; take at most the four best clusters.
    ranked = np.argsort(cluster_scores)[::-1]
    selected = [k for k in ranked
                if cluster_scores[k] > 0.22 and cluster_border_shares[k] < 0.30][:4]
    if selected:
        initial = np.isin(labels, selected).astype(np.uint8)*255
    else:
        initial = color_seed_small.astype(np.uint8)*255
    # Prevent a selected cluster from spreading far away from color evidence.
    seed_support = cv2.dilate(color_seed_small.astype(np.uint8)*255,
                              np.ones((9,9),np.uint8), iterations=2)
    initial = cv2.bitwise_and(initial, seed_support)
    initial = cv2.resize(initial, (w,h), interpolation=cv2.INTER_NEAREST)
    initial = clean_mask(initial, max_components=6, reject_border=True)

    area_ratio = (initial > 0).mean()
    if not (0.02 <= area_ratio <= 0.90):
        # Conservative center rectangle fallback for unusual backgrounds.
        initial[:] = 0
        initial[int(.06*h):int(.94*h), int(.06*w):int(.94*w)] = 255

    if use_grabcut:
        full_seed = banana_color_seeds(rgb)
        hsv_full = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
        bright_neutral = (hsv_full[...,1] < 55) & (hsv_full[...,2] > 115)
        gc = np.full((h,w), cv2.GC_PR_BGD, np.uint8)
        gc[initial > 0] = cv2.GC_PR_FGD
        gc[bright_neutral] = cv2.GC_BGD
        ring2 = max(2, round(min(h,w)*0.02))
        gc[:ring2] = gc[-ring2:] = cv2.GC_BGD
        gc[:,:ring2] = gc[:,-ring2:] = cv2.GC_BGD
        eroded = cv2.erode(cv2.bitwise_and(initial, full_seed),
                             np.ones((5,5),np.uint8), iterations=1)
        if (eroded > 0).sum() > 20: gc[eroded > 0] = cv2.GC_FGD
        try:
            bgd, fgd = np.zeros((1,65),np.float64), np.zeros((1,65),np.float64)
            cv2.grabCut(cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR), gc, None,
                        bgd, fgd, 2, cv2.GC_INIT_WITH_MASK)
            refined = np.isin(gc, [cv2.GC_FGD, cv2.GC_PR_FGD]).astype(np.uint8)*255
            refined = clean_mask(refined, max_components=6, reject_border=True)
            ratio = (refined > 0).mean()
            if 0.02 <= ratio <= 0.90: initial = refined
        except cv2.error:
            pass
    return initial

def process_image(path, segmented=True):
    image = preprocess(read_rgb(path))
    mask = lab_kmeans_segmentation(image) if segmented else np.full(image.shape[:2], 255, np.uint8)
    return image, mask


## 3A. Step-by-step processing of the same five images

Every cell below performs exactly one processing stage and displays its result. The same five images are retained throughout: one from each class plus one additional reproducibly sampled image.

In [ ]:
# STEP 0 — Select five fixed examples and display the originals.
chosen = []
for label_i, label in enumerate(CLASS_ORDER):
    chosen.append(records[records.label == label].sample(1, random_state=RANDOM_STATE+label_i))
preview_records = pd.concat(chosen)
remaining = records.drop(index=preview_records.index)
preview_records = pd.concat([preview_records, remaining.sample(1, random_state=RANDOM_STATE+10)])
preview_records = preview_records.reset_index(drop=True)
preview_labels = preview_records.label.astype(str).tolist()
preview_originals = [read_rgb(path) for path in preview_records.path]

def show_five(images, stage, grayscale=False):
    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    for i, (ax, image, label) in enumerate(zip(axes, images, preview_labels), start=1):
        ax.imshow(image, cmap='gray' if grayscale else None, vmin=0 if grayscale else None,
                  vmax=255 if grayscale else None)
        ax.set_title(f'{i}. {label}')
        ax.axis('off')
    fig.suptitle(stage, fontsize=16)
    plt.tight_layout()
    plt.show()

show_five(preview_originals, 'Step 0 — Original images')


In [ ]:
# STEP 1 — Resize while preserving aspect ratio; pad to TARGET_SIDE × TARGET_SIDE.
preview_resized = [resize_with_padding(image, TARGET_SIDE) for image in preview_originals]
show_five(preview_resized, f'Step 1 — Aspect-preserving resize ({TARGET_SIDE} × {TARGET_SIDE})')


In [ ]:
# STEP 2 — Mild bilateral noise reduction applied to the resized images.
preview_denoised = [cv2.bilateralFilter(image, d=5, sigmaColor=25, sigmaSpace=25)
                     for image in preview_resized]
show_five(preview_denoised, 'Step 2 — Bilateral noise reduction')


In [ ]:
# STEP 3 — CLAHE contrast enhancement on LAB luminance only.
def enhance_lab_luminance(rgb):
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    L, a, b = cv2.split(lab)
    L = cv2.createCLAHE(clipLimit=1.5, tileGridSize=(8,8)).apply(L)
    return cv2.cvtColor(cv2.merge([L, a, b]), cv2.COLOR_LAB2RGB)

preview_enhanced = [enhance_lab_luminance(image) for image in preview_denoised]
show_five(preview_enhanced, 'Step 3 — LAB-luminance CLAHE enhancement')


In [ ]:
# STEP 4 — Generate broad banana-color seed masks (green, yellow, brown, dark peel).
preview_color_seeds = [banana_color_seeds(image) for image in preview_enhanced]
show_five(preview_color_seeds, 'Step 4 — Banana-color foreground seeds', grayscale=True)


In [ ]:
# STEP 5 — LAB K-means color clustering, guided by the seed masks.
def lab_kmeans_raw_mask(rgb):
    h, w = rgb.shape[:2]
    seg_scale = min(1.0, 160/max(h,w))
    sw, sh = max(32, round(w*seg_scale)), max(32, round(h*seg_scale))
    small = cv2.resize(rgb, (sw, sh), interpolation=cv2.INTER_AREA)
    color_seed_small = banana_color_seeds(small) > 0
    lab = cv2.cvtColor(small, cv2.COLOR_RGB2LAB).astype(np.float32)
    yy, xx = np.mgrid[0:sh, 0:sw].astype(np.float32)
    samples = np.column_stack([
        lab[...,0].ravel()/255, (lab[...,1].ravel()-128)/128,
        (lab[...,2].ravel()-128)/128, 0.18*xx.ravel()/sw, 0.18*yy.ravel()/sh
    ]).astype(np.float32)
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.01)
    cv2.setRNGSeed(RANDOM_STATE)
    _, labels, _ = cv2.kmeans(samples, 5, None, criteria, 3, cv2.KMEANS_PP_CENTERS)
    labels = labels.reshape(sh, sw)
    ring = max(2, round(min(sh,sw)*0.08))
    border = np.zeros((sh,sw), bool)
    border[:ring] = border[-ring:] = True
    border[:,:ring] = border[:,-ring:] = True
    center = (((xx-sw/2)/(0.42*sw))**2 + ((yy-sh/2)/(0.42*sh))**2) <= 1
    scores, border_shares = [], []
    for k in range(5):
        cluster = labels == k
        seed_overlap = color_seed_small[cluster].mean() if cluster.any() else 0
        border_share = (labels[border] == k).mean()
        center_share = (labels[center] == k).mean()
        scores.append(2.4*seed_overlap + 0.35*center_share - 1.25*border_share)
        border_shares.append(border_share)
    ranked = np.argsort(scores)[::-1]
    selected = [k for k in ranked if scores[k] > 0.22 and border_shares[k] < 0.30][:4]
    raw = (np.isin(labels, selected) if selected else color_seed_small).astype(np.uint8)*255
    seed_support = cv2.dilate(color_seed_small.astype(np.uint8)*255,
                              np.ones((9,9),np.uint8), iterations=2)
    raw = cv2.bitwise_and(raw, seed_support)
    return cv2.resize(raw, (w,h), interpolation=cv2.INTER_NEAREST)

preview_raw_masks = [lab_kmeans_raw_mask(image) for image in preview_enhanced]
show_five(preview_raw_masks, 'Step 5 — Raw LAB K-means masks', grayscale=True)


In [ ]:
# STEP 6 — Morphological opening/closing, component filtering, and hole filling.
def clean_and_validate_mask(mask):
    cleaned = clean_mask(mask, max_components=6, reject_border=True)
    ratio = (cleaned > 0).mean()
    if not (0.02 <= ratio <= 0.90):
        h, w = cleaned.shape
        cleaned[:] = 0
        cleaned[int(.06*h):int(.94*h), int(.06*w):int(.94*w)] = 255
    return cleaned

preview_clean_masks = [clean_and_validate_mask(mask) for mask in preview_raw_masks]
show_five(preview_clean_masks, 'Step 6 — Morphologically cleaned masks', grayscale=True)


In [ ]:
# STEP 7 — Seeded GrabCut refinement of each cleaned mask.
def refine_mask_grabcut(rgb, initial):
    h, w = initial.shape
    full_seed = banana_color_seeds(rgb)
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    bright_neutral = (hsv[...,1] < 55) & (hsv[...,2] > 115)
    gc = np.full((h,w), cv2.GC_PR_BGD, np.uint8)
    gc[initial > 0] = cv2.GC_PR_FGD
    gc[bright_neutral] = cv2.GC_BGD
    ring = max(2, round(min(h,w)*0.02))
    gc[:ring] = gc[-ring:] = cv2.GC_BGD
    gc[:,:ring] = gc[:,-ring:] = cv2.GC_BGD
    eroded = cv2.erode(cv2.bitwise_and(initial, full_seed),
                         np.ones((5,5),np.uint8), iterations=1)
    if (eroded > 0).sum() > 20:
        gc[eroded > 0] = cv2.GC_FGD
    try:
        bgd, fgd = np.zeros((1,65),np.float64), np.zeros((1,65),np.float64)
        cv2.grabCut(cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR), gc, None,
                    bgd, fgd, 2, cv2.GC_INIT_WITH_MASK)
        refined = np.isin(gc, [cv2.GC_FGD, cv2.GC_PR_FGD]).astype(np.uint8)*255
        refined = clean_mask(refined, max_components=6, reject_border=True)
        if 0.02 <= (refined > 0).mean() <= 0.90:
            return refined
    except cv2.error:
        pass
    return initial

preview_final_masks = [refine_mask_grabcut(image, mask)
                       for image, mask in zip(preview_enhanced, preview_clean_masks)]
show_five(preview_final_masks, 'Step 7 — GrabCut-refined LAB color masks', grayscale=True)

# Redefine the production segmentation wrapper from the same independent stages.
def lab_kmeans_segmentation(rgb, use_grabcut=True):
    raw = lab_kmeans_raw_mask(rgb)
    cleaned = clean_and_validate_mask(raw)
    return refine_mask_grabcut(rgb, cleaned) if use_grabcut else cleaned


In [ ]:
# STEP 8 — Apply each final mask to its enhanced image to obtain the segmented ROI.
preview_rois = [cv2.bitwise_and(image, image, mask=mask)
                for image, mask in zip(preview_enhanced, preview_final_masks)]
show_five(preview_rois, 'Step 8 — Final segmented banana ROIs')


**Stop and inspect all five examples at every step above.** The final masks should retain the complete banana peel while excluding boards, tables, walls, text, and watermarks. If a failure first appears in Step 4, adjust the HSV/LAB seed thresholds; in Step 5, adjust cluster scoring; in Step 6, adjust morphology; or in Step 7, adjust the GrabCut background rule. Visual validation is essential because the dataset has no ground-truth segmentation masks.

In [ ]:
# Feature extraction: normalized color histograms/moments + ripeness ratios + LBP/GLCM + shape.
def normalized_hist(channel, pixels, bins, value_range):
    hist, _ = np.histogram(channel[pixels], bins=bins, range=value_range)
    hist = hist.astype(np.float64)
    return hist / max(hist.sum(), 1)

def extract_features(image, mask):
    fg = mask > 0
    if fg.sum() < 50: fg[:] = True
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    feats, names = [], []

    hist_specs = [('H',hsv[...,0],18,(0,180)), ('S',hsv[...,1],12,(0,256)),
                  ('V',hsv[...,2],12,(0,256)), ('Lab_a',lab[...,1],16,(0,256)),
                  ('Lab_b',lab[...,2],16,(0,256))]
    for prefix, ch, bins, rng in hist_specs:
        vals = normalized_hist(ch, fg, bins, rng)
        feats.extend(vals); names.extend([f'{prefix}_hist_{i}' for i in range(bins)])

    for space_name, arr in [('HSV',hsv), ('LAB',lab)]:
        for c in range(3):
            vals = arr[...,c][fg].astype(float)
            sk = float(skew(vals, bias=False)) if len(vals) > 2 else 0.0
            feats.extend([vals.mean(), vals.std(), np.nan_to_num(sk)])
            names.extend([f'{space_name}{c}_mean', f'{space_name}{c}_std', f'{space_name}{c}_skew'])

    H, S, V = hsv[...,0], hsv[...,1], hsv[...,2]
    valid = max(fg.sum(), 1)
    cue_masks = {
        'green_ratio': fg & (H>=35) & (H<=90) & (S>45) & (V>35),
        'yellow_ratio': fg & (H>=18) & (H<35) & (S>45) & (V>65),
        'brown_ratio': fg & (H>=3) & (H<20) & (S>45) & (V>=25) & (V<200),
        'dark_ratio': fg & (V<70),
        'low_saturation_ratio': fg & (S<45),
    }
    feats.extend([m.sum()/valid for m in cue_masks.values()]); names.extend(cue_masks.keys())

    P, R = 24, 3
    lbp = local_binary_pattern(gray, P, R, method='uniform')
    lbp_hist = normalized_hist(lbp, fg, P+2, (0,P+2))
    feats.extend(lbp_hist); names.extend([f'LBP_{i}' for i in range(P+2)])

    ys, xs = np.where(fg)
    crop = gray[ys.min():ys.max()+1, xs.min():xs.max()+1].copy()
    crop_mask = fg[ys.min():ys.max()+1, xs.min():xs.max()+1]
    crop[~crop_mask] = int(np.median(gray[fg]))
    quant = np.minimum(crop // 16, 15).astype(np.uint8)
    glcm = graycomatrix(quant, distances=[1,3], angles=[0,np.pi/4,np.pi/2,3*np.pi/4],
                        levels=16, symmetric=True, normed=True)
    for prop in ['contrast','dissimilarity','homogeneity','energy','correlation']:
        values = graycoprops(glcm, prop)
        feats.extend([values.mean(), values.std()]); names.extend([f'GLCM_{prop}_mean', f'GLCM_{prop}_std'])

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cnt = max(contours, key=cv2.contourArea) if contours else None
    h, w = mask.shape
    if cnt is None:
        shape = [1,0,1,1,1,0]
    else:
        area = cv2.contourArea(cnt); perimeter = cv2.arcLength(cnt, True)
        x,y,bw,bh = cv2.boundingRect(cnt); hull_area = cv2.contourArea(cv2.convexHull(cnt))
        circularity = 4*np.pi*area/max(perimeter**2,1)
        shape = [fg.mean(), perimeter/(2*(h+w)), area/max(hull_area,1),
                 area/max(bw*bh,1), bw/max(bh,1), circularity]
    shape_names = ['mask_area_ratio','perimeter_norm','solidity','extent','bbox_aspect','circularity']
    feats.extend(shape); names.extend(shape_names)
    return np.asarray(feats, dtype=np.float32), names

ARTIFACT_DIR = Path('/content/banana_svm_artifacts')
ARTIFACT_DIR.mkdir(exist_ok=True)

def build_feature_matrix(segmented=True):
    cache_tag = 'segmented_step_cells_v4' if segmented else 'full_v1'
    cache = ARTIFACT_DIR / f'features_{cache_tag}_{TARGET_SIDE}.joblib'
    if cache.exists():
        payload = joblib.load(cache)
        if payload.get('paths') == records.path.tolist():
            print('Loaded cache:', cache)
            return payload['X'], payload['feature_names']
    vectors, feature_names = [], None
    for path in tqdm(records.path, desc='Segment + extract' if segmented else 'Full-image features'):
        image, mask = process_image(path, segmented=segmented)
        vec, names = extract_features(image, mask)
        vectors.append(vec); feature_names = names
    X = np.vstack(vectors)
    joblib.dump({'X':X, 'feature_names':feature_names, 'paths':records.path.tolist()}, cache)
    return X, feature_names

X_segmented, feature_names = build_feature_matrix(segmented=True)
print('Segmented feature matrix:', X_segmented.shape, 'finite:', np.isfinite(X_segmented).all())


## 4. Split correctly and tune the SVM only on training data

If the published train/test split is found, it is preserved. Hyperparameters are selected using stratified cross-validation inside the training portion only. Scaling is inside the pipeline, preventing leakage from validation/test data. Macro F1 is the optimization metric because every ripeness class should matter equally.

In [ ]:
y_all = records.label.astype(str).to_numpy()
if {'train','test'}.issubset(set(records.split)):
    train_idx = np.flatnonzero(records.split.to_numpy() == 'train')
    test_idx = np.flatnonzero(records.split.to_numpy() == 'test')
    print('Using the dataset-provided train/test split.')
else:
    all_idx = np.arange(len(records))
    train_idx, test_idx = train_test_split(all_idx, test_size=.20, stratify=y_all,
                                            random_state=RANDOM_STATE)
    print('No explicit split found; using a stratified 80/20 split.')

X_train, X_test = X_segmented[train_idx], X_segmented[test_idx]
y_train, y_test = y_all[train_idx], y_all[test_idx]
display(pd.DataFrame({'train':pd.Series(y_train).value_counts(),
                      'test':pd.Series(y_test).value_counts()}).fillna(0).astype(int))

min_class = pd.Series(y_train).value_counts().min()
n_splits = int(min(5, min_class))
assert n_splits >= 2, 'Need at least two training images per class.'
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
pipeline = Pipeline([
    ('scale', StandardScaler()),
    ('svc', SVC(kernel='rbf', class_weight='balanced', probability=True,
                random_state=RANDOM_STATE))
])
param_grid = {
    'svc__C': [1, 10, 50, 100],
    'svc__gamma': ['scale', 0.001, 0.01, 0.1]
}
search = GridSearchCV(pipeline, param_grid, scoring='f1_macro', cv=cv,
                      n_jobs=-1, verbose=1, return_train_score=True)
search.fit(X_train, y_train)
model = search.best_estimator_
print('Best parameters:', search.best_params_)
print('Best CV macro F1:', round(search.best_score_, 4))


In [ ]:
def evaluate(model, X, y, title='SVM'):
    pred = model.predict(X)
    metrics = {
        'accuracy': accuracy_score(y,pred),
        'balanced_accuracy': balanced_accuracy_score(y,pred),
        'macro_f1': f1_score(y,pred,average='macro')
    }
    print(title, {k:round(v,4) for k,v in metrics.items()})
    print(classification_report(y, pred, labels=CLASS_ORDER, zero_division=0))
    cm = confusion_matrix(y, pred, labels=CLASS_ORDER)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='YlGn',
                xticklabels=CLASS_ORDER, yticklabels=CLASS_ORDER)
    plt.xlabel('Predicted'); plt.ylabel('True'); plt.title(title)
    plt.show()
    return metrics, pred

seg_metrics, y_pred = evaluate(model, X_test, y_test, 'LAB segmentation + features + SVM')

# Save everything required to reproduce preprocessing and inference.
bundle = {
    'model': model, 'feature_names': feature_names, 'target_side': TARGET_SIDE,
    'class_order': CLASS_ORDER, 'best_params': search.best_params_,
    'test_metrics': seg_metrics,
    'confidence_method': 'SVC predict_proba with probability=True',
    'preprocessing_config': {
        'bilateral_enabled': True, 'bilateral_d': 5,
        'bilateral_sigma_color': 25, 'bilateral_sigma_space': 25,
        'clahe_clip_limit': 1.5, 'clahe_tile_grid': (8, 8)
    },
    'pipeline_version': 'lab_kmeans_grabcut_features_v1'
}
MODEL_PATH = ARTIFACT_DIR / 'banana_ripeness_color_svm.joblib'
joblib.dump(bundle, MODEL_PATH)
print('Saved model bundle:', MODEL_PATH)


## 5. Required ablation: does segmentation actually help?

This repeats feature extraction with every pixel treated as foreground. For a fair comparison, the no-segmentation SVM receives its own GridSearchCV using the identical search space and training folds. A paired exact McNemar test then checks whether the two models differ beyond their overall scores.

In [ ]:
X_full, _ = build_feature_matrix(segmented=False)
baseline_pipeline = Pipeline([
    ('scale', StandardScaler()),
    ('svc', SVC(kernel='rbf', class_weight='balanced', probability=True,
                random_state=RANDOM_STATE))
])
baseline_search = GridSearchCV(
    baseline_pipeline, param_grid, scoring='f1_macro', cv=cv,
    n_jobs=-1, verbose=1, return_train_score=True
)
baseline_search.fit(X_full[train_idx], y_train)
baseline = baseline_search.best_estimator_
print('Baseline best parameters:', baseline_search.best_params_)
print('Baseline best CV macro F1:', round(baseline_search.best_score_, 4))
base_metrics, base_pred = evaluate(baseline, X_full[test_idx], y_test, 'No-segmentation ablation')
comparison = pd.DataFrame([base_metrics, seg_metrics], index=['No segmentation','LAB segmentation'])
comparison['macro_f1_change_vs_baseline'] = comparison.macro_f1 - base_metrics['macro_f1']
display(comparison.round(4))

# Exact paired McNemar test: uses disagreements on the same test images.
from scipy.stats import binomtest
seg_correct = y_pred == y_test
base_correct = base_pred == y_test
seg_only = int(np.sum(seg_correct & ~base_correct))
base_only = int(np.sum(~seg_correct & base_correct))
discordant = seg_only + base_only
p_value = binomtest(seg_only, discordant, p=.5, alternative='two-sided').pvalue if discordant else 1.0
print({'segmentation_only_correct': seg_only,
       'baseline_only_correct': base_only,
       'exact_mcnemar_p': round(float(p_value), 6)})
print('Statistically significant at 0.05:' , p_value < 0.05)


## 6. Predict an uploaded image

The exact same preprocessing, segmentation, and feature extraction are applied at inference time.

In [ ]:
from google.colab import files
uploaded = files.upload()
for filename, data in uploaded.items():
    upload_path = Path('/content') / filename
    upload_path.write_bytes(data)
    image, mask = process_image(upload_path, segmented=True)
    vector, _ = extract_features(image, mask)
    vector_2d = vector.reshape(1,-1)
    prediction = model.predict(vector_2d)[0]
    probabilities = model.predict_proba(vector_2d)[0]
    probability_table = pd.Series(probabilities, index=model.classes_, name='probability').sort_values(ascending=False)
    confidence = float(probability_table.loc[prediction])
    print(f'Prediction: {prediction}')
    print(f'Model confidence: {confidence:.2%}')
    display(probability_table.to_frame().assign(percentage=lambda x: (100*x.probability).round(2)))
    fig, ax = plt.subplots(1,3,figsize=(12,4))
    ax[0].imshow(read_rgb(upload_path)); ax[0].set_title('Uploaded')
    ax[1].imshow(mask,cmap='gray'); ax[1].set_title('Color mask')
    ax[2].imshow(cv2.bitwise_and(image,image,mask=mask)); ax[2].set_title(f'{prediction} | confidence: {confidence:.2%}')
    for a in ax: a.axis('off')
    plt.show()

# Download this trained bundle and place it beside streamlit_app.py.
# Uncomment after training if you want Colab to download it immediately.
# files.download(str(MODEL_PATH))


## 7. Streamlit GUI deployment

The supplied `streamlit_app.py` applies the same preprocessing, LAB K-means/GrabCut segmentation and feature extraction used during training. After this notebook finishes, download `banana_ripeness_color_svm.joblib` from `/content/banana_svm_artifacts/` and place it in the same directory as the Streamlit application. Install the packages with `pip install -r requirements.txt`, then run `streamlit run streamlit_app.py`. The GUI displays the original image, binary mask, segmented ROI, predicted ripeness class, model confidence and all class probabilities.

> Model confidence is the SVC probability assigned to the predicted class. It describes the classifier's relative certainty; it is not a guarantee that the prediction is correct.

## Reporting checklist

Report: class counts and chosen size from the audit; example masks for all classes; preprocessing parameters; exact train/test protocol; best SVM parameters; accuracy, balanced accuracy and macro F1; confusion matrix; and the segmentation ablation result. Do not describe the LAB/CLAHE image as new data—the SVM is trained only on handcrafted numeric features extracted from the processed images.